In [0]:
import io, os, zipfile
import requests
from pyspark.sql import functions as F

# Começamos com 4 anos; depois é só aumentar a lista
ANOS = [2021, 2022, 2023, 2024]

# Demonstrações que vamos usar (DFC = fluxo de caixa, de onde sai a depreciação para o EBITDA)
DEMONSTRACOES = ["DRE", "BPA", "BPP", "DFC_MI", "DFC_MD"]

URL_DFP = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ano}.zip"
URL_CAD = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"
VOLUME  = "/Volumes/cvm/bronze/arquivos_cvm"

In [0]:
def baixar_dfp(ano):
    destino = f"{VOLUME}/dfp/{ano}"
    os.makedirs(destino, exist_ok=True)
    resposta = requests.get(URL_DFP.format(ano=ano), timeout=300)
    resposta.raise_for_status()                     # para tudo se der erro no download
    with zipfile.ZipFile(io.BytesIO(resposta.content)) as z:
        z.extractall(destino)
        qtd = len(z.namelist())
    print(f"{ano}: {qtd} arquivos extraídos ({len(resposta.content)/1e6:.1f} MB)")

for ano in ANOS:
    baixar_dfp(ano)

# Cadastro das empresas (traz o SETOR de cada uma)
os.makedirs(f"{VOLUME}/cadastro", exist_ok=True)
resposta = requests.get(URL_CAD, timeout=300)
resposta.raise_for_status()
with open(f"{VOLUME}/cadastro/cad_cia_aberta.csv", "wb") as f:
    f.write(resposta.content)
print("Cadastro baixado")

In [0]:
def ler_csv_cvm(caminho):
    return (spark.read
            .option("header", True)
            .option("sep", ";")
            .option("encoding", "ISO-8859-1")        # acentuação do arquivo da CVM
            .csv(caminho)
            .withColumn("_arquivo_origem", F.col("_metadata.file_name"))  # linhagem
            .withColumn("_carregado_em", F.current_timestamp()))

def gravar_bronze(df, tabela):
    (df.write.mode("overwrite")                     # overwrite = rodar 2x não duplica
       .option("overwriteSchema", "true")
       .saveAsTable(tabela))
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

# Demonstrações consolidadas (_con_) de todos os anos de uma vez (o * pega todas as pastas)
for demo in DEMONSTRACOES:
    df = ler_csv_cvm(f"{VOLUME}/dfp/*/dfp_cia_aberta_{demo}_con_*.csv")
    gravar_bronze(df, f"cvm.bronze.dfp_{demo.lower()}_con")

gravar_bronze(ler_csv_cvm(f"{VOLUME}/cadastro/cad_cia_aberta.csv"), "cvm.bronze.cadastro_cia")

In [0]:
%sql
SELECT DENOM_CIA, DT_FIM_EXERC, ORDEM_EXERC, VERSAO, ESCALA_MOEDA, CD_CONTA, DS_CONTA, VL_CONTA
FROM cvm.bronze.dfp_dre_con
WHERE DENOM_CIA LIKE '%AMBEV%' AND CD_CONTA = '3.01'
ORDER BY DT_FIM_EXERC, ORDEM_EXERC